# Gender Bias Exploration — SVC (polynomial kernel)

**Model rationale for ADNI data:**  
SVC with polynomial kernel is the baseline model from the original analysis. It captures non-linear relationships in the 129 neuroimaging + cognitive features while maintaining decision-boundary interpretability via support vectors. The hyperparameters (C=0.1, γ=0.1) were pre-selected via GridSearchCV in the original study.

**6 debiasing methods × 2 imbalance modes = 12 configurations tested.**  
Each configuration trains a baseline model (no debiasing) and a debiased model, then compares them.

In [5]:
%matplotlib inline
from exploration_utils import *
from IPython.display import display, Markdown

In [6]:
# ── Load and prepare data ──
df = load_adni_data()
print('Original shape:', df.shape)
df = drop_metadata(df)
print('After dropping metadata:', df.shape)
display(df.head(2))

Original shape: (757, 147)
After dropping metadata: (757, 129)


,DIAGNOSIS,Sex,age,DXNODEP,DXPARK,DXPDES,DXPCOG,DXPATYP,DXDEP,DXOTHDEM,...,RIGHT_BA36_VOL,RIGHT_BA36_NS,RIGHT_PHC_VOL,RIGHT_PHC_NS,RIGHT_SULCUS_VOL,RIGHT_SULCUS_NS,RIGHT_CA_VOL,RIGHT_CA_NS,RIGHT_HIPP_VOL,RIGHT_HIPP_NS
0,0,0,57.9,0,0,0,0,0,0,0,...,0.099781,12.0,0.040663,7.0,0.017779,18.0,0.094855,20.0,0.153476,20.0
1,0,0,66.4,0,0,0,0,0,0,0,...,0.122244,14.0,0.043713,8.0,0.015407,20.0,0.087257,21.0,0.132635,21.0


In [7]:
# ── BinaryLabelDataset and split ──
dataset = make_bld(df)
dataset_train, dataset_val = split_dataset(dataset)
print(f'Train: {dataset_train.features.shape}, Val: {dataset_val.features.shape}')

# ── Baseline fairness on raw data ──
m_train = BinaryLabelDatasetMetric(dataset_train,
    unprivileged_groups=UNPRIVILEGED_GROUPS, privileged_groups=PRIVILEGED_GROUPS)
m_val = BinaryLabelDatasetMetric(dataset_val,
    unprivileged_groups=UNPRIVILEGED_GROUPS, privileged_groups=PRIVILEGED_GROUPS)
print(f'Training disparate impact = {m_train.disparate_impact():.4f}')
print(f'Validation disparate impact = {m_val.disparate_impact():.4f}')

Train: (499, 128), Val: (258, 128)
Training disparate impact = 0.8232
Validation disparate impact = 1.1675


---
## Baseline (no debiasing)

**Rationale:** Measure the SVC's inherent bias before any intervention. Establishes the reference point.

In [8]:
res_svc_base_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='None')


  SVC | None | SMOTE

  After SMOTE: [405 405] samples
  --- Baseline model ---


  Threshold: 0.0298
  Metrics:
  Balanced accuracy = 0.8554
  Average odds difference = -0.0791
  Disparate impact = 0.9350
  Equal opportunity difference = -0.1290
  Statistical parity difference = -0.0267
  Theil index = 0.0606

  Cases improved by debiasing: 0


IndexError: list index out of range

In [ ]:
res_svc_base_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='None')

---
## DisparateImpactRemover (DIR)

**Rationale:** DIR removes rank correlation between Sex and features by percentile-based redistribution (repair_level=1). Preserves feature distributions and clinical interpretability — no 'black box' debiasing.

In [ ]:
res_svc_dir_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='DIR')

In [ ]:
res_svc_dir_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='DIR')

---
## Reweighing

**Rationale:** Reweighing assigns sample weights to achieve demographic parity in the weighted dataset. Unlike DIR (which modifies features), Reweighing modifies the loss contribution of each instance.

In [ ]:
res_svc_rw_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='Reweighing')

In [ ]:
res_svc_rw_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='Reweighing')

---
## AdversarialDebiasing

**Rationale:** In-processing method: simultaneously trains a classifier and an adversary that predicts Sex from features. The classifier is penalized for features that leak gender information, forcing sex-agnostic representations.

In [ ]:
res_svc_ad_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='AdversarialDebiasing')

In [ ]:
res_svc_ad_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='AdversarialDebiasing')

---
## PrejudiceRemover

**Rationale:** Regularization-based in-processing. Adds a fairness penalty to the loss function that discourages statistical dependence between decisions and the protected attribute.

In [ ]:
res_svc_pr_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='PrejudiceRemover')

In [ ]:
res_svc_pr_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='PrejudiceRemover')

---
## RejectOptionClassifier (post-processing)

**Rationale:** Post-processing: adjusts the decision boundary in regions of uncertainty, favoring the unprivileged group in critical regions. Doesn't modify training — applied after the model predicts.

In [ ]:
res_svc_ro_smote = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=True, use_class_weight=False, debias_name='RejectOption')

In [ ]:
res_svc_ro_cw = run_comparison(
    'svc', dataset_train, dataset_val,
    use_smote=False, use_class_weight=True, debias_name='RejectOption')

---
## Comparison table

In [ ]:
configs = [
    ('Baseline', 'SMOTE', res_svc_base_smote),
    ('Baseline', 'class_weight', res_svc_base_cw),
    ('DIR', 'SMOTE', res_svc_dir_smote),
    ('DIR', 'class_weight', res_svc_dir_cw),
    ('Reweighing', 'SMOTE', res_svc_rw_smote),
    ('Reweighing', 'class_weight', res_svc_rw_cw),
    ('AdversarialDebiasing', 'SMOTE', res_svc_ad_smote),
    ('AdversarialDebiasing', 'class_weight', res_svc_ad_cw),
    ('PrejudiceRemover', 'SMOTE', res_svc_pr_smote),
    ('PrejudiceRemover', 'class_weight', res_svc_pr_cw),
    ('RejectOption', 'SMOTE', res_svc_ro_smote),
    ('RejectOption', 'class_weight', res_svc_ro_cw),
]

rows = [results_row(d, i, r) for d, i, r in configs]
df_svc = pd.DataFrame(rows)
print('=== SVC — Complete Results ===')
display(df_svc.round(4))
save_results('SVC', df_svc)

In [ ]:
# ── Best config by DI improvement (closest to 1.0) while BA doesn't drop below baseline ──
baseline_ba = df_svc.loc[df_svc['Debiasing'] == 'Baseline', 'BA (deb)'].max()
candidates = df_svc[df_svc['BA (deb)'] >= baseline_ba * 0.95].copy()
candidates['DI_dist'] = (candidates['DI (deb)'] - 1.0).abs()
best = candidates.loc[candidates['DI_dist'].idxmin()]
print('Best SVC configuration:')
print(best[['Debiasing', 'Imbalance', 'BA (deb)', 'DI (deb)', 'Improved']].to_string())

# ── Fairness-Accuracy scatter ──
fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(configs)))
for (d, i, _), c in zip(configs, colors):
    row = df_svc[(df_svc['Debiasing'] == d) & (df_svc['Imbalance'] == i)].iloc[0]
    ax.scatter(row['DI (deb)'], row['BA (deb)'], s=120, c=[c], label=f'{d}+{i}')
ax.axvline(1.0, color='gray', ls='--', alpha=0.4, label='DI=1 (parity)')
ax.set_xlabel('Disparate Impact (debiased)')
ax.set_ylabel('Balanced Accuracy (debiased)')
ax.set_title('SVC: Fairness vs Accuracy — all configurations')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

### SVC key findings

- The ideal configuration approaches DI=1.0 (parity) while maintaining BA at or above the baseline.
- SMOTE generally improves BA vs class_weight across most debiasing methods.
- Post-processing (RejectOption) can improve fairness metrics without retraining but may have limited effect if the base model is highly biased.